# GPU-not-needed: evidence, overlap, and CPU-placement economics

Run sequentially using **CutScope analytics**. Successful zero-compute jobs are CPU-placement candidates, not proven portable workloads. All realization and CPU-cost inputs are explicit assumptions. No infrastructure change is performed. Save executed copies in ignored `generated/` or `analytics/notebooks/local/`.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PROJECT_SPEC.md").exists() and (p / "analytics").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open from the CutScope repository")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from analytics.load_data import load_official_data
from analytics.exploration import findings_table
from analytics.scenarios import CpuPlacementPolicy
from analytics.opportunities.gpu_not_needed import gpu_not_needed_candidates
from analytics.pricing import Pricing
from analytics.build_analysis import build_snapshot

PRICE = Pricing(2.50, "cutscope-assumption-v1")
CPU_POLICY = CpuPlacementPolicy(point_realization=.5)  # CPU cost unknown by default
data = load_official_data(os.environ.get("CUTSCOPE_DATA_DIR"))
snapshot = build_snapshot(data, PRICE, cpu_policy=CPU_POLICY)
candidates = gpu_not_needed_candidates(data.jobs)
analysis = snapshot["metadata"]["gpu_not_needed_analysis"]
opportunity = next(o for o in snapshot["opportunities"] if o["id"] == "gpu-not-needed")

## 1. Reproduce the official candidate rule
Require completed state, zero job-average and peak SM, and more than one measured GPU-hour. This partitions successful zero-compute work from unsuccessful zero-compute work. Validate the recomputed IDs against explicitly non-synthetic findings.

In [ ]:
findings = findings_table(data.findings)
official_ids = set(findings.loc[findings.detector_id.eq("rules::gpu-not-needed") & findings.synthetic.eq(False), "job_id"].dropna())
assert set(candidates.id_job) == official_ids
print(f"{len(candidates):,} official candidates consumed {candidates.gpu_hours.sum():,.2f} GPU-hours")
display(candidates.groupby("job_type").agg(jobs=("id_job", "size"), consumed_gpu_hours=("gpu_hours", "sum")))

## 2. Check memory and workload dependencies
Nonzero GPU memory with zero compute may reflect initialization, data storage, a workload dependency, or coarse telemetry. It does not prove GPU compute occurred, and it also does not prove the workload can run without a GPU. Inspect code and output equivalence in a representative pilot.

In [ ]:
print(f"Candidates with nonzero GPU memory: {analysis["candidate_jobs_with_nonzero_gpu_memory"]}; unknown memory: {analysis["candidate_jobs_with_unknown_gpu_memory"]}")
columns = ["id_job", "job_type", "gpu_hours", "gpu_count", "walltime_sec", "sm_util_avg", "sm_util_max", "max_gpu_mem_used", "mem_req_total_mb"]
display(candidates[candidates.max_gpu_mem_used.gt(0)][columns].head(12))
display(candidates[candidates.max_gpu_mem_used.eq(0)][columns].head(12))

## 3. Inspect one job and supporting findings
Change `JOB_ID`. Job cost is observed consumption at the scenario rate, while modeled reclaim is a separate value.

In [ ]:
memory_cases = candidates[candidates.max_gpu_mem_used.gt(0)]
JOB_ID = int((memory_cases if len(memory_cases) else candidates).iloc[0].id_job)
display(data.jobs[data.jobs.id_job.eq(JOB_ID)].T)
display(data.gpus[data.gpus.id_job.eq(JOB_ID)])
for finding in data.findings:
    if (finding.get("metadata") or {}).get("job_id") == JOB_ID:
        display(finding)

## 4. Remove overlap before pricing
Idle-interactive keeps primary precedence, preserving the prior stage. Each remaining eligible job reserves its whole modeled allocation once. Overlap retains supporting evidence on the idle job but adds no extra CPU-placement savings. Primary ownership and display ranking are separate.

In [ ]:
excluded = pd.DataFrame(analysis["exclusions"])
display(excluded)
print(f"Additional modeled jobs: {analysis["modeled_job_count"]}")
refs = {o["id"]: {r["job_id"] for r in o["jobs"]} for o in snapshot["opportunities"]}
assert not refs["idle-interactive"] & refs["gpu-not-needed"]
ledger = snapshot["metadata"]["primary_ledger"]
assert len(ledger) == len({r["job_id"] for r in ledger})
assert len(snapshot["jobs"]) == len({j["job_id"] for j in snapshot["jobs"]})
display(pd.DataFrame(analysis["primary_ledger"]).head())

## 5. Gross GPU resource-value scenarios
High reclaims the lesser of measured and scheduler allocation hours. Low is zero guaranteed reclaim; point assumes 50% realization by default. The values are gross GPU resource value, not net benefit, cash savings, or statistical confidence intervals. Net CPU-placement benefit is unknown unless incremental CPU costs are supplied.

In [ ]:
scenarios = pd.DataFrame({"gpu_hours": analysis["scenarios_gpu_hours"], "gross_gpu_resource_value_usd": analysis["gross_resource_value_usd"]})
display(scenarios)
print("Net benefit:", analysis["net_benefit_usd"])
ax = scenarios.gpu_hours.plot.bar(figsize=(7, 4), title="Additional CPU-placement modeled GPU reclaim")
ax.set_ylabel("GPU-hours in the four-month sample")
plt.tight_layout()
plt.show()

## 6. Incremental CPU cost and runtime sensitivity
Assume a total incremental CPU cost per job-hour for the additional cohort, multiplied by its walltime and a CPU runtime multiplier. This rate is NOT per core and does not price CPU resources already paid for. Inputs below are hypothetical. Negative net benefit is retained; full adoption is not necessarily the best monetary outcome.

In [ ]:
ids = refs["gpu-not-needed"]
selected = data.jobs[data.jobs.id_job.isin(ids)]
walltime_hours = selected.walltime_sec.sum() / 3600
gross_high = analysis["gross_resource_value_usd"]["high"]
sensitivity = []
for multiplier in (1., 1.5, 2.):
    for incremental_rate in (0., .5, 1., 2.5, 5.):
        cpu_cost = walltime_hours * multiplier * incremental_rate
        sensitivity.append({"assumed_incremental_cpu_usd_per_job_hour": incremental_rate, "runtime_multiplier": multiplier, "incremental_cpu_cost_full_adoption": cpu_cost, "net_benefit_full_adoption": gross_high - cpu_cost})
sensitivity = pd.DataFrame(sensitivity)
display(sensitivity)
ax = sensitivity.pivot(index="assumed_incremental_cpu_usd_per_job_hour", columns="runtime_multiplier", values="net_benefit_full_adoption").plot(marker="o", figsize=(8, 4), title="Hypothetical CPU-placement net benefit")
ax.axhline(0, color="black", linewidth=.8)
ax.set(xlabel="Assumed incremental CPU cost ($/job-hour)", ylabel="Net resource-value benefit ($), full adoption")
plt.tight_layout()
plt.show()
print("These curves exclude migration effort, business loss and CPU queue constraints.")

## 7. Validate the configured model
Try other rates or realization fractions. This test verifies the exporter produces the same economics as the notebook, without modifying the default output.

In [ ]:
test_policy = CpuPlacementPolicy(point_realization=.5, incremental_cpu_cost_per_job_hour=1., cpu_runtime_multiplier=1.5)
test_snapshot = build_snapshot(data, PRICE, cpu_policy=test_policy)
test_analysis = test_snapshot["metadata"]["gpu_not_needed_analysis"]
assert abs(test_analysis["incremental_cpu_cost_usd_high_adoption"] - walltime_hours * 1.5) < 1e-6
assert abs(test_analysis["net_benefit_usd"]["high_adoption"] - (gross_high - walltime_hours * 1.5)) < 1e-6
display(test_analysis["net_benefit_usd"])
print("CPU-price assumptions change net producer metadata; public monetary opportunity fields stay explicitly gross GPU resource value.")

## 8. Cost if wrong and handoff
Actual downside and numeric confidence remain unknown. Before changing placement, validate CUDA/library and GPU-memory dependencies, preserve outputs/checkpoints, compare output equivalence and runtime, and confirm CPU queue capacity. Stop the pilot on output or latency regression. Reverting the request cannot undo a missed deadline or already lost work.

In [ ]:
display(opportunity["cost_if_wrong"])
job_ids = {j["job_id"] for j in snapshot["jobs"]}
assert all(o["job_count"] == len(o["jobs"]) and all(r["job_id"] in job_ids for r in o["jobs"]) for o in snapshot["opportunities"])
for key in ("low", "point", "high"):
    ledger_total = sum(row["gpu_hours"][key] for row in ledger)
    producer_total = snapshot["metadata"]["idle_analysis"]["scenarios_gpu_hours"][key] + analysis["scenarios_gpu_hours"][key]
    assert abs(ledger_total - producer_total) < 1e-6
print("Two opportunities reconcile to unique job allocations with complete evidence references. Next: slow-cancel analysis.")